# Skewness (Qiyshiqlik) - Uy vazifasi

**Dataset:** `tips` (Seaborn)

Ushbu notebook'da `tips` dataseti yordamida qiyshiq taqsimotlarni **3 xil usul** bilan tekislaymiz:
1. **Log transformatsiya** (`np.log1p`)
2. **Box-Cox transformatsiya** (`scipy.stats.boxcox`)
3. **Yeo-Johnson transformatsiya** (`sklearn PowerTransformer`)

In [1]:
# Kerakli kutubxonalarni import qilamiz
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from sklearn.preprocessing import PowerTransformer

## 1. Ma'lumotlarni yuklash

In [ ]:
# Seaborn'dan 'tips' datasetini yuklaymiz
df = sns.load_dataset("tips")

print("Dataset shakli:", df.shape)
df.head()

In [ ]:
# Raqamli ustunlarni aniqlaymiz
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
print("Raqamli ustunlar:", list(numeric_cols))

## 2. Barcha raqamli ustunlar uchun qiyshiqlikni hisoblash

In [ ]:
# Har bir ustun uchun qiyshiqlik (skewness) ni ko'rsatamiz
skew_df = pd.DataFrame({
    "Ustun": numeric_cols,
    "Qiyshiqlik": [df[col].skew() for col in numeric_cols]
})

skew_df = skew_df.sort_values(by="Qiyshiqlik", ascending=False)
print(skew_df.to_string(index=False))

In [ ]:
# Barcha raqamli ustunlarning taqsimotini ko'rsatamiz
for col in numeric_cols:
    skew = df[col].skew()
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=col, kde=True, color="steelblue")
    plt.title(f"{col}  |  Qiyshiqlik = {skew:.2f}")
    plt.xlabel(col)
    plt.ylabel("Soni")
    plt.tight_layout()
    plt.show()

## 3. Log Transformatsiya (`np.log1p`)

Log transformatsiya musbat qiyshiq (right-skewed) ma'lumotlar uchun juda samarali.

> `log1p(x) = log(1 + x)` — nol qiymatlarda xato chiqmaslik uchun ishlatiladi.

In [ ]:
# 'total_bill' ustuniga log transformatsiya qo'llaymiz
print("Oldingi qiyshiqlik (total_bill):", round(df["total_bill"].skew(), 4))

df["total_bill_log"] = np.log1p(df["total_bill"])

print("Log transformatsiyadan keyin:", round(df["total_bill_log"].skew(), 4))

In [ ]:
# Log transformatsiya: Oldin va keyin
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["total_bill"], kde=True, ax=ax[0], color="tomato")
ax[0].set_title(f"Asl holat\nQiyshiqlik = {df['total_bill'].skew():.2f}")
ax[0].set_xlabel("total_bill")

sns.histplot(df["total_bill_log"], kde=True, ax=ax[1], color="seagreen")
ax[1].set_title(f"Log Transformatsiyadan keyin\nQiyshiqlik = {df['total_bill_log'].skew():.2f}")
ax[1].set_xlabel("log(1 + total_bill)")

plt.suptitle("Log Transformatsiya: total_bill", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Box-Cox Transformatsiya

Box-Cox transformatsiya optimal `lambda` parametrini avtomatik topib, ma'lumotlarni normal taqsimotga yaqinlashtiradi.

> **Muhim:** Box-Cox faqat **musbat** qiymatlar uchun ishlaydi. Nol yoki manfiy qiymatlar bo'lsa `x + 1` qo'shamiz.

In [ ]:
# 'tip' ustuniga Box-Cox transformatsiya qo'llaymiz
print("Oldingi qiyshiqlik (tip):", round(df["tip"].skew(), 4))

# Box-Cox faqat musbat qiymatlar uchun ishlaydi
df["tip_boxcox"], lam = boxcox(df["tip"] + 1)

print(f"Box-Cox transformatsiyadan keyin (lambda={lam:.4f}):", round(df["tip_boxcox"].skew(), 4))

In [ ]:
# Box-Cox transformatsiya: Oldin va keyin
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["tip"], kde=True, ax=ax[0], color="tomato")
ax[0].set_title(f"Asl holat\nQiyshiqlik = {df['tip'].skew():.2f}")
ax[0].set_xlabel("tip")

sns.histplot(df["tip_boxcox"], kde=True, ax=ax[1], color="royalblue")
ax[1].set_title(f"Box-Cox Transformatsiyadan keyin\nQiyshiqlik = {df['tip_boxcox'].skew():.2f}")
ax[1].set_xlabel("BoxCox(tip)")

plt.suptitle("Box-Cox Transformatsiya: tip", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Yeo-Johnson Transformatsiya

Yeo-Johnson transformatsiya Box-Cox'ning kengaytirilgan versiyasi bo'lib:
- **Musbat va manfiy** qiymatlar bilan ishlaydi
- **Nol qiymatlar** uchun ham to'g'ri ishlaydi

> `sklearn`'ning `PowerTransformer` klassi orqali qo'llaniladi.

In [ ]:
# 'size' ustuniga Yeo-Johnson transformatsiya qo'llaymiz
print("Oldingi qiyshiqlik (size):", round(df["size"].skew(), 4))

pt = PowerTransformer(method="yeo-johnson")
df["size_yeojohnson"] = pt.fit_transform(df[["size"]])

print("Yeo-Johnson transformatsiyadan keyin:", round(df["size_yeojohnson"].skew(), 4))

In [ ]:
# Yeo-Johnson transformatsiya: Oldin va keyin
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["size"], kde=True, ax=ax[0], color="tomato")
ax[0].set_title(f"Asl holat\nQiyshiqlik = {df['size'].skew():.2f}")
ax[0].set_xlabel("size")

sns.histplot(df["size_yeojohnson"], kde=True, ax=ax[1], color="darkorange")
ax[1].set_title(f"Yeo-Johnson Transformatsiyadan keyin\nQiyshiqlik = {df['size_yeojohnson'].skew():.2f}")
ax[1].set_xlabel("YeoJohnson(size)")

plt.suptitle("Yeo-Johnson Transformatsiya: size", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Natijalarni Solishtirish

In [ ]:
# Barcha transformatsiyalar natijasini jadvalda ko'rsatamiz
natijalar = pd.DataFrame({
    "Ustun": ["total_bill", "tip", "size"],
    "Asl Qiyshiqlik": [
        round(df["total_bill"].skew(), 4),
        round(df["tip"].skew(), 4),
        round(df["size"].skew(), 4)
    ],
    "Usul": ["Log (log1p)", "Box-Cox", "Yeo-Johnson"],
    "Yangi Qiyshiqlik": [
        round(df["total_bill_log"].skew(), 4),
        round(df["tip_boxcox"].skew(), 4),
        round(df["size_yeojohnson"].skew(), 4)
    ]
})

natijalar["Yaxshilanish"] = abs(natijalar["Asl Qiyshiqlik"]) - abs(natijalar["Yangi Qiyshiqlik"])
natijalar["Yaxshilanish"] = natijalar["Yaxshilanish"].round(4)

print(natijalar.to_string(index=False))

In [ ]:
# Barcha transformatsiyalarni bitta grafikda solishtirish
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

juftlar = [
    ("total_bill", "total_bill_log", "Log", "seagreen"),
    ("tip", "tip_boxcox", "Box-Cox", "royalblue"),
    ("size", "size_yeojohnson", "Yeo-Johnson", "darkorange"),
]

for i, (asl, yangi, usul, rang) in enumerate(juftlar):
    sns.histplot(df[asl], kde=True, ax=axes[i][0], color="tomato")
    axes[i][0].set_title(f"{asl} — Asl holat\nQiyshiqlik = {df[asl].skew():.2f}")

    sns.histplot(df[yangi], kde=True, ax=axes[i][1], color=rang)
    axes[i][1].set_title(f"{asl} — {usul} keyin\nQiyshiqlik = {df[yangi].skew():.2f}")

plt.suptitle("Barcha Transformatsiyalar Taqqoslash", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Xulosa

| Usul | Qachon ishlatiladi | Cheklovlar |
|---|---|---|
| **Log** | Musbat qiyshiq va musbat qiymatlar | Manfiy/nol qiymatlar bo'lmasligi kerak |
| **Box-Cox** | Kuchli qiyshiqlik, optimal lambda | Faqat musbat qiymatlar |
| **Yeo-Johnson** | Istalgan taqsimot | Cheklov yo'q |

> **Tavsiya:** Agar ma'lumotda manfiy qiymatlar bo'lsa — Yeo-Johnson, faqat musbat bo'lsa — Box-Cox yoki Log usulini tanlang.